<a href="https://colab.research.google.com/github/YashasviShetty112/Supply_Chain_Analytics/blob/main/Supply_Chain_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from google.colab import files
uploaded=files.upload()

Saving DataCoSupplyChainDataset.csv to DataCoSupplyChainDataset.csv


In [4]:
df=pd.read_csv("DataCoSupplyChainDataset.csv",encoding="latin-1")

In [12]:
cols_to_keep=[
    "Order Id",
    "order date (DateOrders)",
    "Customer Segment",
    "Department Name",
    "Market",
    "Order City",
    "Order Country",
    "Product Card Id",
    "Product Name",
    "Product Price",
    "Order Item Quantity",
    "Sales per customer",
    "Benefit per order",
    "Late_delivery_risk",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
]


In [13]:
print(df.columns.tolist())

['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Status', 'shipping date (DateOrde

In [14]:
df_clean = df[cols_to_keep].copy()

In [15]:
df_clean["order date (DateOrders)"]=pd.to_datetime(df_clean["order date (DateOrders)"])

In [16]:
df_clean=df_clean.dropna(subset=["Order Id", "Product Card Id", "Sales per customer"]
)

In [18]:
sku_revenue = (
    df_clean.groupby(["Product Card Id", "Product Name"])["Sales per customer"]
    .sum()
    .reset_index()
    )

In [19]:
sku_revenue = sku_revenue.sort_values(by="Sales per customer", ascending=False)

In [20]:
sku_revenue["Cumulative_Revenue"] = sku_revenue["Sales per customer"].cumsum()
sku_revenue["Cumulative_Percentage"] = (
    sku_revenue["Cumulative_Revenue"] / sku_revenue["Sales per customer"].sum()
) * 100

In [22]:
def assign_abc(pct):
  if pct <= 80:
    return "A" # Top 80% revenue items (Strict control)
  elif pct <= 95:
    return "B" # Next 15% revenue items (Moderate control)
  else:
    return "C" # Bottom 5% revenue items (Loose control)


sku_revenue["ABC_Class"] = sku_revenue["Cumulative_Percentage"].apply(
    assign_abc
)

In [23]:
df_clean = df_clean.merge(
    sku_revenue[["Product Card Id", "ABC_Class"]], on="Product Card Id", how="left"
)

In [25]:
df_clean.to_csv("Clean_Supply_Chain.csv",index="False")
sku_revenue.to_csv("sku_abc_classification.csv",index="False")